# Task 2: Text Chunking, Embedding, and Vector Store Indexing

This notebook creates a stratified sample from CFPB complaints, chunks narratives, generates embeddings using **all-MiniLM-L6-v2**, and indexes them in **ChromaDB** with metadata.

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os
from typing import List, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

c:\Users\Eyasu\Documents\10th\intelligent-complaint-analysis-for-financial-services-week7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# # Create required directories
# os.makedirs('vector_store', exist_ok=True)
# os.makedirs('data/processed', exist_ok=True)

## Step 1: Load Filtered Dataset & Stratified Sampling

In [5]:
print("Loading filtered complaints dataset...")
df = pd.read_csv('../data/processed/filtered_complaints.csv')

product_categories = [
    'Credit card',
    'Consumer Loan',
    'Bank account or service',
    'Money transfer, virtual currency'
]

df = df[df['Product'].isin(product_categories)].copy()
print(f"Loaded {len(df):,} complaints across {len(product_categories)} categories.")

Loading filtered complaints dataset...
Loaded 105,013 complaints across 4 categories.


In [7]:
# Proportional stratified sampling
total_samples = 12000
category_sizes = df['Product'].value_counts()
proportions = category_sizes / len(df)
sample_sizes = (proportions * total_samples).round().astype(int)

# Adjust rounding differences
diff = total_samples - sample_sizes.sum()
if diff != 0:
    sample_sizes[sample_sizes.idxmax()] += diff

print("Sampling strategy:")
for cat, size in sample_sizes.items():
    print(f"  {cat}: {size} samples ({size / total_samples * 100:.1f}%)")

df_sample = df.groupby('Product', group_keys=False).apply(
    lambda x: x.sample(n=sample_sizes.get(x.name, 0), random_state=42)
).reset_index(drop=True)

print(f"Sampled dataset shape: {len(df_sample):,}")
df_sample.to_csv('../data/processed/sampled_complaints.csv', index=False)

Sampling strategy:
  Credit card: 9218 samples (76.8%)
  Bank account or service: 1701 samples (14.2%)
  Consumer Loan: 1081 samples (9.0%)
Sampled dataset shape: 12,000


C:\Users\Eyasu\AppData\Local\Temp\ipykernel_25492\3659870057.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('Product', group_keys=False).apply(


## Step 2: Text Chunking

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_list = []
for _, row in df_sample.iterrows():
    narrative = row['cleaned_narrative']
    if pd.isna(narrative) or not narrative.strip():
        continue
    chunks = splitter.split_text(narrative)
    for idx, chunk in enumerate(chunks):
        chunks_list.append({
            'complaint_id': row['Complaint ID'],
            'product_category': row['Product'],
            'chunk_text': chunk,
            'chunk_index': idx,
            'total_chunks': len(chunks)
        })

df_chunks = pd.DataFrame(chunks_list)
print(f"Generated {len(df_chunks):,} chunks from {len(df_sample):,} complaints")

df_chunks.to_csv('../data/processed/sampled_chunks.csv', index=False)

Generated 37,087 chunks from 12,000 complaints


## Step 3: Embedding Generation

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

chunk_texts = df_chunks['chunk_text'].tolist()
embeddings = model.encode(chunk_texts, normalize_embeddings=True)

print(f"Embeddings generated: {embeddings.shape}")

c:\Users\Eyasu\Documents\10th\intelligent-complaint-analysis-for-financial-services-week7\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Eyasu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## Step 4: Vector Store Indexing (ChromaDB)

In [ ]:
client = chromadb.PersistentClient(path='./vector_store')
collection_name = 'cfpb_sample_complaints'

try:
    collection = client.get_collection(name=collection_name)
    print(f"Loaded existing collection '{collection_name}'")
except:
    collection = client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    print(f"Created collection '{collection_name}'")

In [ ]:
ids = [f"{row.complaint_id}_chunk_{row.chunk_index}" for row in df_chunks.itertuples()]
metadatas = [
    {
        'complaint_id': row.complaint_id,
        'product_category': row.product_category,
        'chunk_index': row.chunk_index,
        'total_chunks': row.total_chunks,
        'text_preview': row.chunk_text[:100]
    }
    for row in df_chunks.itertuples()
]

batch_size = 1000
for i in range(0, len(ids), batch_size):
    collection.add(
        ids=ids[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size].tolist(),
        metadatas=metadatas[i:i+batch_size]
    )
    print(f"Indexed batch {i//batch_size + 1}")

print(f"Total vectors indexed: {collection.count}")

## Optional: Retrieval Sanity Check

In [ ]:
query = model.encode(["billing issues with credit card"])
results = collection.query(
    query_embeddings=query.tolist(),
    n_results=3,
    include=['metadatas', 'distances']
)

for meta, dist in zip(results['metadatas'][0], results['distances'][0]):
    print(f"Distance: {dist:.3f} | {meta['product_category']} | {meta['text_preview']}...")